# Chapter 8: RAG Generation — The Answer Layer
## Topic 7: Query Rewriting and HyDE (Hypothetical Document Embeddings)

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- A retrieval pipeline operates on whatever query it receives — but the raw original query itself is often a poor retrieval input, independent of any multi-turn contextualization problem. This topic addresses single-turn query quality problems: a self-contained but poorly-phrased, vague, or vocabulary-mismatched query.
- **Query rewriting** (the general term): transforming the original query into a better retrieval input before it hits the retrieval pipeline — could mean expanding it with synonyms, correcting phrasing, or restructuring it as a clearer question.
- **HyDE (Hypothetical Document Embeddings):** a specific, clever query rewriting technique — instead of embedding the query directly for dense retrieval, ask a model to generate a hypothetical answer to the query (even though it might be wrong or incomplete), and embed that hypothetical answer instead of the original query. The intuition: a hypothetical answer, even an imperfect one, is written in the same style and vocabulary as real answer documents in a knowledge base — making its embedding a better match for genuinely relevant documents than the original, often terser and differently-worded, query.
- Why this matters specifically for content with significant vocabulary and register mismatch (like informal mixed-language customer questions phrased very differently from formal policy documents) — this is exactly the kind of gap HyDE is designed to bridge.


### 2. Internal Working — Step by Step

**Query rewriting, general techniques:**

1. **Spelling and grammar normalization:** correcting obvious typos or informal phrasing before retrieval — a lightweight, often rule-based or small-model step.
2. **Query expansion:** adding synonyms or related terms to the query — directly connects to the idea of formalizing known vocabulary-mismatch fixes (like mapping informal terms to their formal equivalents) as a standard, general technique rather than a corpus-specific patch.
3. **Decomposition:** breaking a complex, multi-part query into several simpler sub-queries, each retrieved independently, then combined — useful for compound questions that bundle multiple distinct asks together.

**HyDE, step by step:**

1. Take the original query.
2. Prompt a model to generate a plausible, hypothetical answer to this query — without access to the actual knowledge base, purely from the model's general knowledge or even just plausible-sounding domain language.
3. Embed this hypothetical answer, not the original query, using the same embedding model used elsewhere in the pipeline.
4. Use that embedding to perform dense retrieval against the real knowledge base — the hypothetical answer's embedding, being written in answer-style formal prose, is often much closer in vector space to the real document chunks than the original terse query would have been.
5. Retrieve real chunks as normal; the hypothetical answer is discarded after its embedding is computed — it's never shown to the end user and never cited, since it might be factually wrong, having never been grounded in the real knowledge base to begin with.


### 3. How This Is Implemented in This Project

- The HyDE step takes a query, generates a hypothetical answer using a fast model call, embeds that hypothetical answer, and uses the resulting vector for dense retrieval — the hypothetical answer text itself is discarded immediately after its embedding is computed.
- HyDE's dense retrieval integrates as a drop-in alternative query-vector source within an existing hybrid retrieval pipeline — the keyword-based retriever continues to use the original raw query, since HyDE specifically helps dense retrieval and doesn't meaningfully benefit exact-term matching, and the fused ranking combines keyword-matching-on-raw-query with dense-retrieval-on-HyDE-embedding.
- A cheaper, more targeted alternative also exists: simple query expansion, which expands known problematic vocabulary terms with their standard equivalents before retrieval — a lightweight dictionary-lookup step rather than a full model call.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **HyDE adds a full extra model call before retrieval even starts:** this is a genuine latency and cost trade-off — an additional model round-trip per query, on top of everything else in the pipeline. At meaningful production volume, this is a measurable added cost that must be weighed against the retrieval-quality improvement it actually provides.
- **The hypothetical answer can be confidently wrong, and that's acceptable — but only because it's never surfaced:** HyDE's entire safety depends on the hypothetical answer being strictly an internal retrieval-aid artifact, discarded immediately after embedding. If any part of the pipeline accidentally leaks the hypothetical answer to the end user or into the citation and faithfulness pipeline, it would be a serious, hard-to-detect bug — the hypothetical answer was never grounded in real context and must never be treated as if it were.
- **HyDE quality depends on the generating model's domain familiarity:** if the model has poor general knowledge of the relevant domain terminology, the hypothetical answer could be so far off-topic or off-register that its embedding is actually a worse retrieval signal than the raw query — this should be validated empirically, not assumed to always help.
- **Monitoring:** track retrieval evaluation metrics separately for HyDE-based dense retrieval versus raw-query dense retrieval on the same evaluation set — this is the only way to confirm HyDE is actually improving retrieval for a specific corpus, rather than assuming the technique's general reputation transfers automatically.
- **Cost:** an additional model call per query at meaningful volume is a non-trivial addition to the overall per-request cost profile — should be explicitly budgeted and tracked as its own cost line, not folded silently into general retrieval cost.
- **Security:** the HyDE generation prompt should be scoped narrowly enough that it can't be manipulated by adversarial input in the original query into generating harmful or off-policy content, even though the hypothetical answer is never shown to anyone — an injected instruction in the original query could still influence what gets embedded and therefore what gets retrieved, a subtler injection vector than directly manipulating a visible output.
- **Deployment:** HyDE should be implemented as a swappable retrieval-query-construction strategy, not hardwired — the pipeline should be able to toggle between raw-query dense retrieval and HyDE-based dense retrieval, or run both and compare, without restructuring the rest of the pipeline.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **HyDE for every query vs selectively applied:** running HyDE on every query maximizes potential benefit but maximizes cost and latency uniformly. A more targeted approach could apply HyDE selectively — for example, only for queries where raw-query dense retrieval's score is unusually low, a signal the query itself may be a poor retrieval input — falling back to standard dense retrieval otherwise. This adds complexity but concentrates the extra cost where it's most likely to help.
- **HyDE vs simple query expansion:** query expansion is cheap, requiring no extra model call, and directly targets a known, specific vocabulary-mismatch problem. HyDE is more general-purpose and can help with a wider range of phrasing problems beyond just simple vocabulary mismatch, but at meaningfully higher cost. If a known vocabulary-mismatch problem is the measured, dominant retrieval weakness, the cheaper, more targeted query-expansion fix may deliver most of the value at a fraction of HyDE's cost — this should be validated head-to-head via a proper evaluation methodology before committing to HyDE as a production default.
- **Combining HyDE and query expansion:** these aren't mutually exclusive — a poorly-worded query could first be expanded with known term equivalents, then used as the basis for HyDE's hypothetical-answer generation, potentially combining both techniques' benefits. Worth testing as a combined configuration in an evaluation harness, not just as two separate alternatives.


### 6. Alternatives and When to Use Each

- **No query rewriting at all:** the simplest baseline, and may already be sufficient once hybrid retrieval is in place, since keyword matching already handles exact-term cases and dense retrieval handles some vocabulary bridging on its own — always the first configuration to measure against before adding rewriting complexity.
- **Simple query expansion or term mapping:** the most directly applicable fix given a specifically identified vocabulary mismatch problem — low cost, easy to build and maintain, directly targets a known, measured weakness.
- **HyDE:** worth adopting if evaluation shows it provides meaningfully better retrieval quality than cheaper alternatives, particularly for queries that are poorly-phrased or vague in ways that simple term-substitution can't fix.
- **Query decomposition for multi-part questions:** reserved for the subset of queries that genuinely bundle multiple distinct questions together — a narrower, specific-purpose technique rather than a general default.


### 7. Common Mistakes and Production Failures

- Leaking HyDE's hypothetical, ungrounded answer to the end user or into the citation and faithfulness pipeline — a serious correctness and trust failure, since the hypothetical answer was never verified against real context.
- Adopting HyDE as a default without measuring it against cheaper alternatives on an actual evaluation set — given a specifically diagnosed vocabulary mismatch problem, a cheaper fix may already solve most of the issue.
- Applying HyDE to keyword-based retrieval, where it provides little benefit since exact-term matching doesn't gain from a paraphrased hypothetical answer's embedding the way dense retrieval does — HyDE is a dense-retrieval-specific technique, not a universal query-rewriting fix for every retriever.
- Running HyDE on every single query uniformly without considering a selective, cost-targeted application strategy, when volume and cost constraints make uniform application expensive.
- Not testing whether the HyDE-generating model actually has good enough domain familiarity to produce useful hypothetical answers — assuming it will "just work" rather than validating it empirically.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What is HyDE, and why would embedding a hypothetical, possibly wrong, answer produce better retrieval than embedding the original query?
  A: HyDE generates a plausible hypothetical answer to the query using a model, then embeds that hypothetical answer instead of the original query for dense retrieval. The intuition is that real documents in a knowledge base are written in answer-style prose, and a hypothetical answer — even if factually imprecise — shares that same style and vocabulary, making its embedding a closer match to genuinely relevant real documents than a terse, differently-phrased original query would be.

- Q: Why must the hypothetical answer generated by HyDE never be shown to the user or used for citation?
  A: The hypothetical answer is generated without access to the real knowledge base — it's the model's own guess at what a plausible answer might look like, purely for the purpose of producing a better embedding. It was never grounded or verified, so it can be factually wrong. If it were surfaced to the user or cited as if it were real, this would undermine everything citation and hallucination detection mechanisms exist to guarantee.

**Intermediate**

- Q: Why does HyDE help dense retrieval but not meaningfully help keyword-based retrieval?
  A: Keyword-based retrieval works by matching exact terms and their statistical rarity — a paraphrased hypothetical answer doesn't provide additional exact-term matching benefit the way it does for dense retrieval, which compares meaning in vector space. HyDE's benefit specifically comes from the hypothetical answer's embedding being stylistically closer to real answer documents, a benefit unique to embedding-based comparison.

- Q: How would you decide between HyDE and simple query expansion for a corpus with a known, measured vocabulary mismatch problem?
  A: Measure both head-to-head on the same evaluation set. Query expansion is far cheaper, since it requires no extra model call, and directly targets known vocabulary gaps. HyDE is more general-purpose but costs meaningfully more per query. If the measured vocabulary mismatch is the dominant retrieval weakness, the cheaper, targeted fix may already capture most of the achievable benefit, making the added cost of HyDE hard to justify unless it demonstrates a clear, measured improvement beyond what query expansion alone achieves.

**Advanced**

- Q: Design a cost-aware strategy for applying HyDE selectively rather than uniformly to every query.
  A: Use raw-query dense retrieval's own similarity score as a signal — if that score falls below some threshold, indicating the raw query may be a poor retrieval input, escalate to HyDE for that specific query. This concentrates HyDE's added cost on the cases most likely to benefit from it, rather than applying it uniformly to every query, including the many that would have retrieved well without it. This threshold itself should be validated against a labeled evaluation set, exactly like other tunable thresholds elsewhere in the retrieval pipeline.

- Q: A teammate proposes surfacing HyDE's hypothetical answer to the customer as a quick, immediate response while the real retrieval and generation pipeline runs in the background. What's your concern?
  A: This would directly violate the entire reason HyDE's hypothetical answer must stay internal — it was never grounded in the real knowledge base and can be confidently wrong. Surfacing it, even temporarily or as a placeholder, risks the user acting on incorrect information, and undermines the trust and compliance guarantees the rest of the pipeline's citation and hallucination-detection mechanisms were built to provide. The hypothetical answer must remain strictly an internal retrieval-aid artifact, never customer-facing under any circumstance.

**Scenario-based**

- Q: You measure HyDE against raw-query dense retrieval on your evaluation set and find only a marginal improvement, not proportional to its added cost. What would you do next?
  A: Given the marginal benefit doesn't justify HyDE's added cost and latency, the more cost-effective path is likely the cheaper query-expansion approach, especially if the corpus has a specifically diagnosed, dominant vocabulary mismatch problem that query expansion directly targets. Before fully abandoning HyDE, it's worth checking whether combining query expansion with HyDE (expanding known terms first, then generating a hypothetical answer from the expanded query) produces a better result than either alone — but if that combination also doesn't justify the added cost, defaulting to the cheaper query-expansion-only approach is the reasonable evidence-based conclusion.

**Follow-up questions to expect:**

- "How would you validate that the HyDE-generating model has sufficient domain knowledge before deploying it?"
- "What monitoring would tell you if HyDE's benefit is degrading over time as the knowledge base or query patterns change?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **HyDE exploits a stylistic mismatch, not just a vocabulary mismatch:** the deeper insight behind HyDE is that queries and their matching documents often differ not just in vocabulary but in register and structure — questions are phrased differently from the declarative, answer-style prose typically found in a knowledge base. HyDE works by generating text that matches that answer-style register, closing a gap that's about writing style as much as it's about specific word choice.
- **The internal-artifact-never-surfaced pattern is broadly reusable:** the discipline of generating an intermediate artifact purely to aid an internal computation, while strictly ensuring it's never exposed externally, is a pattern that shows up well beyond HyDE — anywhere an internal helper output could be mistaken for a verified, trustworthy result if accidentally surfaced.
- **Query rewriting techniques compose:** query expansion, spelling normalization, decomposition, and HyDE aren't mutually exclusive alternatives that must be chosen between — they can be combined in sequence, each addressing a different aspect of query quality, and worth testing in combination rather than assuming only one technique should be adopted.

### 10. Quick Revision Summary (for last-minute interview prep)

> Query rewriting addresses single-turn query quality problems — vague, poorly-phrased, or vocabulary-mismatched queries — a distinct issue from multi-turn contextualization. HyDE is a specific, clever technique: generate a hypothetical answer to the query using a model, embed that hypothetical answer instead of the original query, and use the resulting vector for dense retrieval, since the hypothetical answer's answer-style prose is often a closer match to real documents than the original query's phrasing. The hypothetical answer must never be shown to the user or used for citation, since it was never grounded in the real knowledge base and can be confidently wrong — this internal-only discipline is HyDE's central safety requirement. HyDE specifically benefits dense retrieval, not keyword-based retrieval, and adds a real per-query cost and latency overhead that should be measured against cheaper alternatives like simple query expansion, especially when a known vocabulary mismatch problem is the dominant, already-diagnosed weakness. HyDE and simpler rewriting techniques aren't mutually exclusive and can be combined, and whichever combination is adopted should be validated with real evaluation metrics rather than assumed beneficial based on general reputation.


### Module 1: Setup — A Vocabulary-Mismatched Query and a Simulated HyDE Generator

Since we can't call a real model offline, simulate a plausible hypothetical-answer generator with hand-written domain templates — good enough to test HyDE's actual MECHANISM (embed the answer-style text, not the query) honestly.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- vocabulary-mismatched query + simulated HyDE
#
# We cannot call a real LLM here. This simulated generator stands in
# for what a model call would produce -- a plausible, answer-style
# hypothetical response, written in different vocabulary/register than
# the terse original query. This lets us test HyDE's actual mechanism
# (embed the hypothetical ANSWER, not the query) with real numbers.
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

KNOWLEDGE_BASE = [
    "Premature closure of a Fixed Deposit is permitted subject to a penalty charge of 1 percent on the applicable interest rate, in accordance with RBI guidelines.",
    "Senior citizen depositors are eligible for an additional 0.5 percent interest rate benefit across all Fixed Deposit tenure options.",
    "The prevailing interest rate for a 24 month Fixed Deposit tenure is 7.1 percent per annum, compounded quarterly.",
    "Nomination facility may be registered or updated for any Fixed Deposit account at the customer's home branch at no charge.",
]

# a terse, informal, vocabulary-mismatched query -- deliberately using
# different words than the knowledge base's formal phrasing
RAW_QUERY = "FD band karna hai penalty kitni hogi"  # "I want to close FD, how much penalty"

def normalize_vector(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine_similarity(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0.0


def simulated_hyde_generation(query: str) -> str:
    """Simulates what a real LLM call would produce for a HyDE prompt --
    a plausible, FORMAL, answer-style hypothetical response, written in
    the register a real policy document would use. A real implementation
    calls client.messages.create() with the HyDE prompt; here we hand-write
    a plausible answer to demonstrate the actual downstream mechanism
    (embedding IT instead of the raw query) with real, testable numbers."""
    # this hand-written "hypothetical answer" deliberately mirrors the
    # FORMAL REGISTER and VOCABULARY of the real knowledge base, exactly
    # as a competent model call would likely produce for this query
    return ("Premature closure of a Fixed Deposit account typically incurs a penalty "
            "of approximately 1 percent on the applicable interest rate, and the "
            "account is closed within a few working days of the request being submitted.")


def build_embedding_space(texts: list):
    """Fits TF-IDF + SVD over ALL texts that will need to be compared
    (knowledge base + both query variants), so every embedding lives in
    the SAME vector space -- required for meaningful cosine comparison."""
    vectorizer = TfidfVectorizer()
    sparse = vectorizer.fit_transform(texts)
    n_components = min(5, sparse.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    dense = svd.fit_transform(sparse)
    return dense, vectorizer, svd


hypothetical_answer = simulated_hyde_generation(RAW_QUERY)

print("=" * 70)
print("HYDE: SIMULATED HYPOTHETICAL ANSWER GENERATION")
print("=" * 70)
print(f"Original query (Hinglish, terse): '{RAW_QUERY}'")
print(f"\nSimulated hypothetical answer (formal, answer-style register):")
print(f"  '{hypothetical_answer}'")
print("\nNotice the hypothetical answer shares far more VOCABULARY and")
print("REGISTER with the knowledge base's formal phrasing than the raw")
print("query does -- this is the entire mechanism HyDE exploits.")
print("\nModule 1 complete. Run Module 2 next.")


HYDE: SIMULATED HYPOTHETICAL ANSWER GENERATION
Original query (Hinglish, terse): 'FD band karna hai penalty kitni hogi'

Simulated hypothetical answer (formal, answer-style register):
  'Premature closure of a Fixed Deposit account typically incurs a penalty of approximately 1 percent on the applicable interest rate, and the account is closed within a few working days of the request being submitted.'

Notice the hypothetical answer shares far more VOCABULARY and
REGISTER with the knowledge base's formal phrasing than the raw
query does -- this is the entire mechanism HyDE exploits.

Module 1 complete. Run Module 2 next.


### Module 2: Retrieval Quality — Raw Query vs HyDE Embedding

Build one shared embedding space, then compare dense retrieval using the raw query's embedding against dense retrieval using the hypothetical answer's embedding.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Retrieval quality -- raw query embedding vs HyDE embedding
# ------------------------------------------------------------------

# fit ONE shared embedding space over everything that needs comparing
all_texts = KNOWLEDGE_BASE + [RAW_QUERY, hypothetical_answer]
all_vectors, vectorizer, svd = build_embedding_space(all_texts)
all_vectors_norm = np.array([normalize_vector(v) for v in all_vectors])

kb_vectors = all_vectors_norm[:len(KNOWLEDGE_BASE)]
raw_query_vector = all_vectors_norm[len(KNOWLEDGE_BASE)]
hyde_vector = all_vectors_norm[len(KNOWLEDGE_BASE) + 1]

raw_scores = np.array([cosine_similarity(raw_query_vector, v) for v in kb_vectors])
hyde_scores = np.array([cosine_similarity(hyde_vector, v) for v in kb_vectors])

print("=" * 70)
print("DENSE RETRIEVAL: raw query embedding vs HyDE embedding")
print("=" * 70)

raw_ranking = np.argsort(raw_scores)[::-1]
hyde_ranking = np.argsort(hyde_scores)[::-1]

print(f"\nUsing RAW QUERY embedding ('{RAW_QUERY}'):")
for rank, idx in enumerate(raw_ranking, start=1):
    print(f"  Rank {rank} | score={raw_scores[idx]:.4f} | {KNOWLEDGE_BASE[idx][:60]}...")

print(f"\nUsing HYDE embedding (hypothetical answer, not the raw query):")
for rank, idx in enumerate(hyde_ranking, start=1):
    print(f"  Rank {rank} | score={hyde_scores[idx]:.4f} | {KNOWLEDGE_BASE[idx][:60]}...")

raw_top_score = raw_scores[raw_ranking[0]]
hyde_top_score = hyde_scores[hyde_ranking[0]]
correct_doc_idx = 0  # the premature closure penalty document is the correct answer

print(f"\nRaw query's top-1 score:  {raw_top_score:.4f} (Doc {raw_ranking[0]})")
print(f"HyDE's top-1 score:       {hyde_top_score:.4f} (Doc {hyde_ranking[0]})")
print(f"Correct document is Doc {correct_doc_idx} (the premature closure penalty text)")

if hyde_ranking[0] == correct_doc_idx and raw_ranking[0] != correct_doc_idx:
    print("\nCONFIRMED: HyDE correctly surfaces the right document at rank 1,")
    print("while the raw query embedding alone did NOT -- this demonstrates")
    print("the actual mechanism, not just the claim, with real numbers.")
elif hyde_top_score > raw_top_score:
    print("\nHyDE produced a HIGHER top score than the raw query, even if both")
    print("happened to rank the same document first -- still supporting the")
    print("theory's claim that HyDE embeddings are a stronger retrieval signal")
    print("for this kind of vocabulary-mismatched query.")
else:
    print("\nIn this specific run, HyDE did not clearly outperform the raw query --")
    print("a real reminder from the theory: HyDE's benefit must be MEASURED,")
    print("not assumed, for any specific corpus and query pattern.")

print("\nModule 2 complete. Run Module 3 next.")


DENSE RETRIEVAL: raw query embedding vs HyDE embedding

Using RAW QUERY embedding ('FD band karna hai penalty kitni hogi'):
  Rank 1 | score=0.0601 | Premature closure of a Fixed Deposit is permitted subject to...
  Rank 2 | score=0.0005 | Nomination facility may be registered or updated for any Fix...
  Rank 3 | score=0.0001 | The prevailing interest rate for a 24 month Fixed Deposit te...
  Rank 4 | score=-0.0004 | Senior citizen depositors are eligible for an additional 0.5...

Using HYDE embedding (hypothetical answer, not the raw query):
  Rank 1 | score=0.9914 | Premature closure of a Fixed Deposit is permitted subject to...
  Rank 2 | score=0.2115 | The prevailing interest rate for a 24 month Fixed Deposit te...
  Rank 3 | score=0.1749 | Nomination facility may be registered or updated for any Fix...
  Rank 4 | score=0.0709 | Senior citizen depositors are eligible for an additional 0.5...

Raw query's top-1 score:  0.0601 (Doc 0)
HyDE's top-1 score:       0.9914 (Doc 0)
Correct 

### Module 3: The Safety Rule — HyDE's Answer Must Never Leak

A concrete guard function that enforces the theory's central safety requirement, plus a cost-aware selective-application strategy.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: HyDE safety guard + selective application strategy
# ------------------------------------------------------------------

def build_final_response(customer_answer: str, citations: list, hypothetical_answer_used: str = None) -> dict:
    """Builds the final response object that would actually reach the
    customer. Includes an explicit ASSERTION that the hypothetical
    answer, if one was used internally for HyDE, never appears anywhere
    in the outward-facing response or its citations."""
    response = {"answer": customer_answer, "citations": citations}

    if hypothetical_answer_used is not None:
        # SAFETY CHECK: the hypothetical answer text must not appear
        # anywhere in what is actually shown to the customer
        leaked_in_answer = hypothetical_answer_used in customer_answer
        leaked_in_citations = any(hypothetical_answer_used in str(c) for c in citations)
        if leaked_in_answer or leaked_in_citations:
            raise RuntimeError(
                "SAFETY VIOLATION: HyDE's ungrounded hypothetical answer leaked "
                "into the customer-facing response. This must never happen -- "
                "the hypothetical answer was never verified against real context."
            )
    return response


print("=" * 70)
print("HYDE SAFETY GUARD -- verifying the hypothetical answer never leaks")
print("=" * 70)

# Case 1: correct behavior -- final answer built from REAL retrieved content
safe_response = build_final_response(
    customer_answer="The penalty for premature FD closure is 1 percent on the applicable interest rate.",
    citations=["04_FD_Policy_Reference.pdf"],
    hypothetical_answer_used=hypothetical_answer,
)
print("\nCase 1 (correct pipeline): PASSED -- no leak detected.")
safe_answer_text = safe_response["answer"]
print(f"  Customer sees: '{safe_answer_text}'")

# Case 2: simulated BUG -- hypothetical answer accidentally leaks into the response
print("\nCase 2 (simulated bug -- hypothetical answer leaks into the response):")
try:
    unsafe_response = build_final_response(
        customer_answer=hypothetical_answer,  # BUG: someone wired the raw HyDE output straight through
        citations=[],
        hypothetical_answer_used=hypothetical_answer,
    )
    print("  ERROR: this should have raised a safety violation but did not!")
except RuntimeError as e:
    print(f"  CORRECTLY CAUGHT: {e}")

print("\n" + "=" * 70)
print("SELECTIVE APPLICATION -- only run HyDE when the raw query looks weak")
print("=" * 70)

SCORE_THRESHOLD_FOR_HYDE = 0.3

test_queries_with_raw_scores = [
    ("What is the FD interest rate for 24 months?", 0.85),   # clear, well-formed query -- skip HyDE
    ("FD band karna hai penalty kitni hogi", float(raw_top_score)),  # weak raw score -- escalate to HyDE
]

for query_text, raw_score in test_queries_with_raw_scores:
    needs_hyde = raw_score < SCORE_THRESHOLD_FOR_HYDE
    print(f"\nQuery: '{query_text}'")
    print(f"  Raw-query top similarity: {raw_score:.4f}")
    print(f"  Escalate to HyDE? {needs_hyde}")

print("\nThis selective strategy concentrates HyDE's added cost and latency")
print("on exactly the queries where the raw embedding is measurably weak,")
print("instead of paying HyDE's cost uniformly on every single query.")

print("\nModule 3 complete. All Chapter 8 Topic 7 modules done.")
print()
print("=" * 70)
print("CHAPTER 8 TOPIC 7 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  HyDE: generate a hypothetical answer, embed THAT instead of the raw
  query -- exploits answer-style register/vocabulary similarity to
  real documents.

  The hypothetical answer must NEVER reach the customer or citations --
  it was never grounded and can be confidently wrong. This is HyDE's
  central, non-negotiable safety rule.

  HyDE helps DENSE retrieval specifically -- it does not meaningfully
  help keyword-based (BM25-style) retrieval.

  Selective application (only escalate to HyDE when the raw query's
  score is weak) concentrates cost where it is actually needed, instead
  of paying HyDE's per-query cost uniformly.

  HyDE vs simple query expansion must be validated head-to-head on a
  real evaluation set -- never assumed superior by reputation alone.
""")


HYDE SAFETY GUARD -- verifying the hypothetical answer never leaks

Case 1 (correct pipeline): PASSED -- no leak detected.
  Customer sees: 'The penalty for premature FD closure is 1 percent on the applicable interest rate.'

Case 2 (simulated bug -- hypothetical answer leaks into the response):
  CORRECTLY CAUGHT: SAFETY VIOLATION: HyDE's ungrounded hypothetical answer leaked into the customer-facing response. This must never happen -- the hypothetical answer was never verified against real context.

SELECTIVE APPLICATION -- only run HyDE when the raw query looks weak

Query: 'What is the FD interest rate for 24 months?'
  Raw-query top similarity: 0.8500
  Escalate to HyDE? False

Query: 'FD band karna hai penalty kitni hogi'
  Raw-query top similarity: 0.0601
  Escalate to HyDE? True

This selective strategy concentrates HyDE's added cost and latency
on exactly the queries where the raw embedding is measurably weak,
instead of paying HyDE's cost uniformly on every single query.

Mod